In [ ]:
!pip install transformers datasets scikit-learn torch pandas numpy

In [ ]:
!git clone https://github.com/suu990901/chatgpt-comparison-detection-HC3-Plus.git
%cd chatgpt-comparison-detection-HC3-Plus

In [ ]:
import pandas as pd
import json

file_path = "data/en/train.jsonl"

data = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

print("Dataset Loaded Successfully!")
print("Shape:", df.shape)

In [ ]:
# Check original label count
print("Original label count:")
print(df["label"].value_counts())

# Separate labels
human_df = df[df["label"] == 0]
ai_df = df[df["label"] == 1]

# Check if enough samples are available
if len(human_df) >= 25000 and len(ai_df) >= 25000:
    human_sample = human_df.sample(n=25000, random_state=42)
    ai_sample = ai_df.sample(n=25000, random_state=42)

    balanced_df = pd.concat([human_sample, ai_sample])
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\nBalanced dataset created successfully!")
    print(balanced_df["label"].value_counts())
    print("Final shape:", balanced_df.shape)

else:
    print("\nNot enough samples for 25,000 each.")
    print("Human available:", len(human_df))
    print("AI available:", len(ai_df))

In [ ]:
from sklearn.model_selection import train_test_split

X = balanced_df["text"]
y = balanced_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 30000
MAX_LEN = 200

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(X_train)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

print("Padding completed!")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),

    LSTM(128, dropout=0.3, recurrent_dropout=0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=5,
    batch_size=64
)

In [ ]:
loss, accuracy = model.evaluate(
    X_test_pad,
    y_test,
    verbose=1
)

print(f"Test Accuracy: {accuracy*100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

pred_probs = model.predict(X_test_pad)

preds = (pred_probs > 0.5).astype(int)

print(
    classification_report(
        y_test,
        preds,
        target_names=["Human", "AI Generated"]
    )
)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

model.save(
    "/content/drive/MyDrive/lstm_hc3plus_model.keras"
)

print("Model saved!")

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

model = load_model(
    "/content/drive/MyDrive/lstm_hc3plus_model.keras"
)

def predict_text(text):

    if len(text.split()) < 10:
        print("⚠️ Enter at least 10 words")
        return

    seq = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(
        seq,
        maxlen=MAX_LEN
    )

    prob = model.predict(padded, verbose=0)[0][0]

    label = "🤖 AI Generated" if prob > 0.5 else "👤 Human Written"

    confidence = max(prob, 1-prob)

    print("Prediction:", label)
    print(f"Confidence: {confidence*100:.2f}%")

In [ ]:
while True:

    text = input(
        "\nEnter text (or type quit): "
    )

    if text.lower() == "quit":
        break

    predict_text(text)